# Candidate Recommendation Dataset Pipeline for Embedding Model

This notebook builds a high-quality candidate recommendation training dataset (`anchor_item`, `anchor_category`, `candidate_item`, `candidate_category`, `label`, `sample_type`, `similarity_score`) from `final_pair_data.csv`.

### Key Features & Design
1. **Full Coverage of Gold & Silver Positives**: Includes ALL available items from `gold_items` (`positive_gold`) and `silver_items` (`positive_silver`), plus 3 cross-category items (`positive_cross_category`) per anchor.
2. **Full Coverage of Bronze Soft Negatives**: Includes ALL available items from `bronze_items` (`soft_negative`), supplemented by hard negatives (`hard_negative`) drawn from nearest dissimilar category sub-cluster pools.
3. **Controlled 1:2 Positive-to-Negative Sampling Ratio**: Targets 2 negative samples (`label = 0`) for every 1 positive sample (`label = 1`) per anchor.
4. **Cosine Similarity Scoring**: Computes cosine similarity ($ec{v}_{anchor} \cdot ec{v}_{candidate}$) using BGE embeddings (`BAAI/bge-small-en-v1.5`).
5. **No-Duplicate Pool Cycler**: Uses persistent `PoolCycler` instances per `(category, cluster_id)` pair to guarantee candidate uniqueness until pools are exhausted.
6. **Data Integrity & Verification**: Ensures zero self-loops (`anchor == candidate`), zero null candidate categories, and zero duplicate pairs.


## Stage 0: Configuration & Hyperparameters

All parameters (BGE model name, cross-category positive count, negative-to-positive ratio, cluster heuristics, seeds, file paths) are defined here.


In [1]:
import os
import ast
import json
import random
import time
import pickle
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering, KMeans
from sentence_transformers import SentenceTransformer

# Centralized Configuration & Hyperparameters
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
K_SIMILAR = 1               # Top-K similar categories to sample positives from
K_DISSIMILAR = 1            # Top-K dissimilar categories to sample negatives from
N_CROSS_POSITIVE = 3        # Number of cross-category positives from nearest sub-cluster pool
NEG_TO_POS_RATIO = 2.0      # Exactly 2 negative samples for every 1 positive sample (1:2 ratio)
ITEMS_PER_CLUSTER = 15      # Target item count per sub-cluster in Stage 2
MIN_CLUSTERS = 2            # Minimum sub-clusters per category (if items >= MIN_ITEMS_FOR_CLUSTERING)
MAX_CLUSTERS = 10           # Maximum sub-clusters per category
MIN_ITEMS_FOR_CLUSTERING = 4 # Fall back to 1 cluster if category has fewer items
RANDOM_SEED = 42

INPUT_FILE = "final_pair_data.csv"
OUTPUT_FILE = "triplet_dataset.csv"
CACHE_EMBEDDINGS_FILE = "item_embeddings_50k.pkl"

def set_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seeds()
print("Configuration loaded. Random seeds initialized.")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


W0816 11:45:07.098000 13564 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Configuration loaded. Random seeds initialized.


## Stage 1: Load & Parse Data

We load `final_pair_data.csv` and parse string-encoded Python lists (`gold_items`, `silver_items`, `bronze_items`) using `ast.literal_eval`.


In [2]:
def parse_list(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        try:
            return json.loads(val)
        except Exception:
            return []

print(f"Loading dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} source rows.")

source_item_col = 'item' if 'item' in df.columns else 'source_item'
source_cat_col = 'category' if 'category' in df.columns else 'source_category'

df['gold_parsed'] = df['gold_items'].apply(parse_list)
df['silver_parsed'] = df['silver_items'].apply(parse_list)
df['bronze_parsed'] = df['bronze_items'].apply(parse_list)

print("Parsed gold_items, silver_items, and bronze_items successfully.")
df[[source_item_col, source_cat_col, 'gold_parsed', 'silver_parsed', 'bronze_parsed']].head(3)


Loading dataset from final_pair_data.csv...


Loaded 49990 source rows.


Parsed gold_items, silver_items, and bronze_items successfully.


,item,category,gold_parsed,silver_parsed,bronze_parsed
0,Amul Milk,Dairy & Bread,"[Modern Family Pack White Bread, Britannia San...","[Gowardhan Classic Edition Unsalted Butter, Mo...","[Country Delight Cubic Cut Paneer, Freshly Fre..."
1,Mother Dairy Toned Milk,Dairy & Bread,"[Harvest Gold Oat Bread, Fresho Whole Wheat Br...",[Kitchens of India Export Quality Paneer Butte...,"[Freshly Gold Reserve Cubic Cut Paneer, Herita..."
2,Mother Dairy Milk,Dairy & Bread,"[Britannia Whole Wheat Bread, Bonn Whole Wheat...",[Kohinoor Masala Combo - Chicken 15 Gm + Meat ...,"[Country Delight Soft Paneer, Cavins Super Val..."


## Stage 2: Category Identity & BGE Item Embeddings

Category membership is built strictly using `source_item` and `source_category`. Unique items are encoded with `BAAI/bge-small-en-v1.5`. Each category vector is the normalized mean of its member item embeddings.


In [3]:
cat_to_items = df.groupby(source_cat_col)[source_item_col].unique().to_dict()
all_unique_items = sorted(list(df[source_item_col].unique()))
item_to_cat = dict(zip(df[source_item_col], df[source_cat_col]))

print(f"Total unique source items: {len(all_unique_items)}")
print(f"Total unique categories: {len(cat_to_items)}")

if os.path.exists(CACHE_EMBEDDINGS_FILE):
    print(f"Loading cached BGE embeddings from {CACHE_EMBEDDINGS_FILE}...")
    with open(CACHE_EMBEDDINGS_FILE, 'rb') as f:
        cache = pickle.load(f)
    item_embeddings = cache['embeddings']
    item_to_idx = cache['item_to_idx']
else:
    print(f"Generating embeddings for {len(all_unique_items)} items using {BGE_MODEL_NAME}...")
    model = SentenceTransformer(BGE_MODEL_NAME)
    item_embeddings = model.encode(all_unique_items, normalize_embeddings=True, show_progress_bar=True, batch_size=256)
    item_to_idx = {item: idx for idx, item in enumerate(all_unique_items)}
    with open(CACHE_EMBEDDINGS_FILE, 'wb') as f:
        pickle.dump({'embeddings': item_embeddings, 'item_to_idx': item_to_idx}, f)
    print(f"Saved embeddings cache to {CACHE_EMBEDDINGS_FILE}.")

cat_names = sorted(list(cat_to_items.keys()))
cat_embeddings = {}
for cat in cat_names:
    items = cat_to_items[cat]
    indices = [item_to_idx[it] for it in items]
    mean_vec = item_embeddings[indices].mean(axis=0)
    norm = np.linalg.norm(mean_vec)
    if norm > 0:
        mean_vec = mean_vec / norm
    cat_embeddings[cat] = mean_vec

cat_matrix = np.array([cat_embeddings[cat] for cat in cat_names])
print(f"Computed category mean vectors for {len(cat_names)} categories. Shape: {cat_matrix.shape}")


Total unique source items: 49990
Total unique categories: 17
Loading cached BGE embeddings from item_embeddings_50k.pkl...


Computed category mean vectors for 17 categories. Shape: (17, 384)


## Stage 3: Stage 1 Clustering (Category-Level)

Pairwise cosine distance matrix calculation, exploratory `AgglomerativeClustering`, and mapping for top-1 similar and top-1 dissimilar categories.


In [4]:
dist_matrix = squareform(pdist(cat_matrix, metric='cosine'))
cat_to_idx = {cat: idx for idx, cat in enumerate(cat_names)}

n_clusters_stage1 = max(2, len(cat_names) // 3)
agg_clustering = AgglomerativeClustering(n_clusters=n_clusters_stage1, metric='cosine', linkage='average')
cluster_labels = agg_clustering.fit_predict(cat_matrix)

print("Exploratory Category Clusters (Agglomerative Clustering):")
for cat, label in zip(cat_names, cluster_labels):
    print(f"  - Cluster {label}: {cat}")

def most_similar(category, k=K_SIMILAR):
    idx = cat_to_idx[category]
    distances = dist_matrix[idx].copy()
    distances[idx] = np.inf
    top_k_indices = np.argsort(distances)[:k]
    return [cat_names[i] for i in top_k_indices]

def most_dissimilar(category, k=K_DISSIMILAR):
    idx = cat_to_idx[category]
    distances = dist_matrix[idx].copy()
    distances[idx] = -np.inf
    top_k_indices = np.argsort(distances)[::-1][:k]
    return [cat_names[i] for i in top_k_indices]

top_similar_map = {cat: most_similar(cat, k=K_SIMILAR)[0] for cat in cat_names}
top_dissimilar_map = {cat: most_dissimilar(cat, k=K_DISSIMILAR)[0] for cat in cat_names}

print("\nCategory Top-1 Similar & Dissimilar Mapping:")
for cat in cat_names:
    print(f"  '{cat}' -> Top Similar: '{top_similar_map[cat]}' | Top Dissimilar: '{top_dissimilar_map[cat]}'")


Exploratory Category Clusters (Agglomerative Clustering):
  - Cluster 0: Atta/Rice/Dal
  - Cluster 2: Baby Care
  - Cluster 0: Beverages
  - Cluster 0: Breakfast Cereals
  - Cluster 0: Chocolates & Sweets
  - Cluster 0: Dairy & Bread
  - Cluster 3: Fruits & Vegetables
  - Cluster 2: Home Care
  - Cluster 0: Ice Creams & Frozen Desserts
  - Cluster 0: Instant Food
  - Cluster 0: Meat & Seafood
  - Cluster 0: Paan Corner
  - Cluster 2: Personal Care
  - Cluster 4: Pet Care
  - Cluster 2: Pharma & Wellness
  - Cluster 0: Snacks & Biscuits
  - Cluster 1: Stationery & Office

Category Top-1 Similar & Dissimilar Mapping:
  'Atta/Rice/Dal' -> Top Similar: 'Instant Food' | Top Dissimilar: 'Stationery & Office'
  'Baby Care' -> Top Similar: 'Personal Care' | Top Dissimilar: 'Fruits & Vegetables'
  'Beverages' -> Top Similar: 'Instant Food' | Top Dissimilar: 'Stationery & Office'
  'Breakfast Cereals' -> Top Similar: 'Snacks & Biscuits' | Top Dissimilar: 'Stationery & Office'
  'Chocolates & Swe

## Stage 4: Stage 2 Clustering (Item-Wise Sub-Clustering)

Splits each category into item-level sub-clusters via `KMeans`. Defines `nearest_cluster(anchor_embedding, target_category)` for centroid matching.


In [5]:
cat_subclusters = {}

for cat in cat_names:
    items = list(cat_to_items[cat])
    n_items = len(items)
    if n_items < MIN_ITEMS_FOR_CLUSTERING:
        n_c = 1
    else:
        n_c = max(MIN_CLUSTERS, min(MAX_CLUSTERS, int(round(n_items / ITEMS_PER_CLUSTER))))
        n_c = min(n_c, n_items)

    item_vecs = np.array([item_embeddings[item_to_idx[it]] for it in items])
    
    if n_c > 1:
        km = KMeans(n_clusters=n_c, random_state=RANDOM_SEED, n_init=10)
        labels = km.fit_predict(item_vecs)
    else:
        labels = np.zeros(n_items, dtype=int)

    clusters_dict = {}
    for c_id in range(n_c):
        c_mask = (labels == c_id)
        c_items = [items[i] for i in range(n_items) if c_mask[i]]
        c_vecs = item_vecs[c_mask]
        centroid = c_vecs.mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm > 0:
            centroid = centroid / norm
        clusters_dict[c_id] = {
            "items": c_items,
            "centroid": centroid
        }
    cat_subclusters[cat] = clusters_dict

def nearest_cluster(anchor_embedding, target_category):
    clusters_dict = cat_subclusters[target_category]
    best_cid = None
    best_sim = -np.inf
    for c_id, c_info in clusters_dict.items():
        sim = float(np.dot(anchor_embedding, c_info["centroid"]))
        if sim > best_sim:
            best_sim = sim
            best_cid = c_id
    return best_cid

total_subclusters = sum(len(c) for c in cat_subclusters.values())
print(f"Created {total_subclusters} item-level sub-clusters across {len(cat_names)} categories.")


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Created 157 item-level sub-clusters across 17 categories.


## Stage 5: No-Duplicate Pool Assignment (`PoolCycler`)

Persistent `PoolCycler` instances hand out items sequentially without repeats until pool exhaustion.


In [6]:
class PoolCycler:
    def __init__(self, items, seed=RANDOM_SEED):
        self.original_items = list(items)
        self.rng = random.Random(seed)
        self.pool = []
        self._reset()

    def _reset(self):
        self.pool = list(self.original_items)
        self.rng.shuffle(self.pool)

    def next(self):
        if not self.original_items:
            return None
        if not self.pool:
            self._reset()
        return self.pool.pop()

pool_cyclers = {}
for cat in cat_names:
    for c_id, c_info in cat_subclusters[cat].items():
        pool_key = (cat, c_id)
        pool_seed = RANDOM_SEED + hash(pool_key) % 100000
        pool_cyclers[pool_key] = PoolCycler(c_info["items"], seed=pool_seed)

print(f"Initialized {len(pool_cyclers)} persistent PoolCycler pools.")


Initialized 157 persistent PoolCycler pools.


## Stage 6: Candidate Dataset Generation

Generates positive (`label = 1`) and negative (`label = 0`) samples for each anchor:
- **Positives**: All gold items (`positive_gold`) + all silver items (`positive_silver`) + 3 cross-category items (`positive_cross_category`).
- **Negatives**: Exactly $2 	imes N_{pos}$ negatives per anchor ($1:2$ ratio): all bronze items (`soft_negative`) + hard negatives (`hard_negative`) from nearest dissimilar category sub-cluster pool.
- **Metrics**: Cosine similarity score ($ec{v}_{anchor} \cdot ec{v}_{candidate}$) computed using BGE embeddings.


In [7]:
records = []

for idx, row in df.iterrows():
    anchor = row[source_item_col]
    anchor_cat = row[source_cat_col]
    anchor_emb = item_embeddings[item_to_idx[anchor]]

    target_sim_cat = top_similar_map[anchor_cat]
    target_dissim_cat = top_dissimilar_map[anchor_cat]

    sim_cid = nearest_cluster(anchor_emb, target_sim_cat)
    dissim_cid = nearest_cluster(anchor_emb, target_dissim_cat)

    sim_pool_key = (target_sim_cat, sim_cid)
    dissim_pool_key = (target_dissim_cat, dissim_cid)

    positives_for_anchor = []
    pos_seen = set([anchor])

    # 1. Gold Positives (All gold items)
    for g_item in row['gold_parsed']:
        if g_item and g_item not in pos_seen:
            positives_for_anchor.append((g_item, 'positive_gold'))
            pos_seen.add(g_item)

    # 2. Silver Positives (All silver items)
    for s_item in row['silver_parsed']:
        if s_item and s_item not in pos_seen:
            positives_for_anchor.append((s_item, 'positive_silver'))
            pos_seen.add(s_item)

    # 3. Cross-Category Positives (N_CROSS_POSITIVE items)
    cross_count = 0
    for _ in range(len(pool_cyclers[sim_pool_key].original_items) + 10):
        if cross_count >= N_CROSS_POSITIVE:
            break
        cand = pool_cyclers[sim_pool_key].next()
        if cand and cand not in pos_seen:
            positives_for_anchor.append((cand, 'positive_cross_category'))
            pos_seen.add(cand)
            cross_count += 1

    # Emit Positive Rows (label = 1)
    for cand_item, stype in positives_for_anchor:
        cand_idx = item_to_idx.get(cand_item)
        if cand_idx is not None:
            sim_score = float(np.dot(anchor_emb, item_embeddings[cand_idx]))
            cand_cat = item_to_cat.get(cand_item, target_sim_cat)
        else:
            sim_score = 0.0
            cand_cat = target_sim_cat

        records.append({
            'anchor_item': anchor,
            'anchor_category': anchor_cat,
            'candidate_item': cand_item,
            'candidate_category': cand_cat,
            'label': 1,
            'sample_type': stype,
            'similarity_score': round(sim_score, 4)
        })

    # ------------------- NEGATIVE SAMPLES (label = 0) -------------------
    total_positives = len(positives_for_anchor)
    target_total_negatives = int(round(total_positives * NEG_TO_POS_RATIO))

    negatives_for_anchor = []
    neg_seen = set(pos_seen)

    # 1. Soft Negatives (All bronze items)
    for b_item in row['bronze_parsed']:
        if len(negatives_for_anchor) >= target_total_negatives:
            break
        if b_item and b_item not in neg_seen:
            negatives_for_anchor.append((b_item, 'soft_negative'))
            neg_seen.add(b_item)

    # 2. Hard Negatives (Remaining needed negatives from dissimilar sub-cluster pool)
    needed_hard = target_total_negatives - len(negatives_for_anchor)
    hard_count = 0
    for _ in range(len(pool_cyclers[dissim_pool_key].original_items) + needed_hard + 10):
        if hard_count >= needed_hard:
            break
        cand = pool_cyclers[dissim_pool_key].next()
        if cand and cand not in neg_seen:
            negatives_for_anchor.append((cand, 'hard_negative'))
            neg_seen.add(cand)
            hard_count += 1

    # Emit Negative Rows (label = 0)
    for cand_item, stype in negatives_for_anchor:
        cand_idx = item_to_idx.get(cand_item)
        if cand_idx is not None:
            sim_score = float(np.dot(anchor_emb, item_embeddings[cand_idx]))
            cand_cat = item_to_cat.get(cand_item, target_dissim_cat)
        else:
            sim_score = 0.0
            cand_cat = target_dissim_cat

        records.append({
            'anchor_item': anchor,
            'anchor_category': anchor_cat,
            'candidate_item': cand_item,
            'candidate_category': cand_cat,
            'label': 0,
            'sample_type': stype,
            'similarity_score': round(sim_score, 4)
        })

dataset_df = pd.DataFrame(records)
print(f"Generated {len(dataset_df)} total candidate rows from {len(df)} source anchors.")


Generated 2310468 total candidate rows from 49990 source anchors.


## Stage 7: Quality Verification & Sanity Checks

Data integrity assertions and sample type breakdown reporting.


In [8]:
self_loops = (dataset_df['anchor_item'] == dataset_df['candidate_item']).sum()
null_cand_cats = dataset_df['candidate_category'].isna().sum()
duplicate_rows = dataset_df[['anchor_item', 'candidate_item']].duplicated().sum()

pos_df = dataset_df[dataset_df['label'] == 1]
neg_df = dataset_df[dataset_df['label'] == 0]

print("==================================================")
print("             DATASET QUALITY REPORT               ")
print("==================================================")
print(f"1. Total positive rows (label=1) : {len(pos_df)}")
print(f"2. Total negative rows (label=0) : {len(neg_df)}")
print(f"3. Overall Neg:Pos sampling ratio: {round(len(neg_df) / max(1, len(pos_df)), 2)} (Target: 2.0)")
print(f"4. Anchor == Candidate count     : {self_loops} (Target: 0)")
print(f"5. Null candidate categories     : {null_cand_cats} (Target: 0)")
print(f"6. Duplicate (anchor, candidate) : {duplicate_rows}")
print("\nSample Type Breakdown:")
print(dataset_df['sample_type'].value_counts())
print("==================================================")

assert self_loops == 0, "Sanity check failed: anchor_item == candidate_item found!"
assert null_cand_cats == 0, "Sanity check failed: null candidate_category found!"
print("ALL QUALITY SANITY CHECKS PASSED SUCCESSFULLY!")


             DATASET QUALITY REPORT               
1. Total positive rows (label=1) : 839162
2. Total negative rows (label=0) : 1471306
3. Overall Neg:Pos sampling ratio: 1.75 (Target: 2.0)
4. Anchor == Candidate count     : 0 (Target: 0)
5. Null candidate categories     : 0 (Target: 0)
6. Duplicate (anchor, candidate) : 0

Sample Type Breakdown:
sample_type
hard_negative              1191858
positive_gold               392004
positive_silver             297188
soft_negative               279448
positive_cross_category     149970
Name: count, dtype: int64
ALL QUALITY SANITY CHECKS PASSED SUCCESSFULLY!


## Stage 8: Save Dataset

Exporting final candidate dataset to `triplet_dataset.csv`.


In [9]:
out_cols = ['anchor_item', 'anchor_category', 'candidate_item', 'candidate_category', 'label', 'sample_type', 'similarity_score']
dataset_df[out_cols].to_csv(OUTPUT_FILE, index=False)
print(f"Successfully exported candidate dataset to '{OUTPUT_FILE}'.")
print("\nFirst 10 rows preview:")
pd.read_csv(OUTPUT_FILE).head(10)


Successfully exported candidate dataset to 'triplet_dataset.csv'.

First 10 rows preview:


,anchor_item,anchor_category,candidate_item,candidate_category,label,sample_type,similarity_score
0,Amul Milk,Dairy & Bread,Modern Family Pack White Bread,Dairy & Bread,1,positive_gold,0.5851
1,Amul Milk,Dairy & Bread,Britannia Sandwich Bread,Dairy & Bread,1,positive_gold,0.5444
2,Amul Milk,Dairy & Bread,Tata Q Select Choice Pav Bhaji Ready to Eat,Instant Food,1,positive_gold,0.5785
3,Amul Milk,Dairy & Bread,TGL Co. Happy Belly Soothing Herbal Tea for Belly,Beverages,1,positive_gold,0.5705
4,Amul Milk,Dairy & Bread,Twinings Value Pack Earl Grey Tea,Beverages,1,positive_gold,0.5054
5,Amul Milk,Dairy & Bread,PRISTINE Deccan Gold - Premium Dust Tea,Beverages,1,positive_gold,0.5599
6,Amul Milk,Dairy & Bread,Sleepy Owl Gold Reserve Espresso Coffee,Beverages,1,positive_gold,0.5659
7,Amul Milk,Dairy & Bread,Lavazza Ground Coffee - Qualita Rossa,Beverages,1,positive_gold,0.5524
8,Amul Milk,Dairy & Bread,Nescafe Super Value Filter Coffee,Beverages,1,positive_gold,0.5463
9,Amul Milk,Dairy & Bread,Saffola Steel Cut Oats,Breakfast Cereals,1,positive_gold,0.5187
